# 09. KOBIS 외부 변수 추가 (스크린 수 · 상영 횟수)

**대회 규칙**: "공공 데이터와 같이 누구나 얻을 수 있고 법적 제약이 없는 외부 데이터 허용"
**출처**: 영화진흥위원회 KOBIS 오픈API · 일별 박스오피스 (`kobis_fetch.py`로 수집 → `kobis_movies.csv`)

| 변수 | 설명 |
| --- | --- |
| `k_in_top10` | KOBIS 일별 박스오피스 상위 10위에 한 번이라도 들었는지 |
| `k_open_scrn` · `k_open_show` | 개봉일(또는 개봉 후 첫 진입일) 스크린 수 · 상영 횟수 |
| `k_max_scrn` · `k_max_show` | 상영 기간 중 최대 스크린 수 · 상영 횟수 |
| `k_days_top10` · `k_best_rank` | 상위 10위에 든 일수 · 최고 순위 |

- **쓰지 않은 것**: 일별 · 누적 관객수, 매출액 → 정답(최종 관객수)을 거의 그대로 담고 있어서 제외 (수집 단계에서 저장도 안 함)
- **한계**: 하루 상위 10편까지만 제공 → 작은 영화는 값이 없음 (0 + `k_in_top10`=0으로 처리)
- **주의**: 스크린 수는 개봉 후에 정해지는 정보 → 개봉 전 예측에는 못 쓰는 변수 (발표 때 밝히기)

| 단계 | 내용 |
| --- | --- |
| 1 | KOBIS 영화를 train · test에 매칭 (제목 + 개봉일) |
| 2 | 외부 변수와 관객수의 관계 확인 |
| 3 | 08 최종 방식(6개 부스팅 · 트리 + TabPFN, NNLS 가중 평균)에 외부 변수를 넣고 같은 5-Fold CV로 비교 |

## 0. 설정

In [1]:
import os
import re
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.optimize import nnls
from sklearn import set_config
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from ngboost import NGBRegressor
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

from features import make_features

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
kobis = pd.read_csv(os.path.join(DATA_DIR, 'kobis_movies.csv'))
y = train['box_off_num']
hit = y >= 1_000_000
rmse = lambda a, b: mean_squared_error(a, b) ** 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
CACHE_DIR = 'cache_09'
os.makedirs(CACHE_DIR, exist_ok=True)
print('KOBIS 영화', len(kobis), '편 | device', DEVICE)

KOBIS 영화 1371 편 | device cpu


C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. KOBIS 매칭 (제목 + 개봉일)
제목에서 공백 · 특수문자를 지우고 비교 → 같은 제목이 여러 개면 개봉일이 가장 가까운 것 (30일 이내만 인정)

In [2]:
norm = lambda s: re.sub(r'[^0-9a-zA-Z가-힣]', '', str(s)).lower()
kobis['key'] = kobis['movieNm'].map(norm)
kobis['open_dt'] = pd.to_datetime(kobis['openDt'], errors='coerce')
K_COLS = ['open_scrn', 'open_show', 'max_scrn', 'max_show', 'days_top10', 'best_rank']


def match(df):
    out = []
    rel = pd.to_datetime(df['release_time'])
    for title, r in zip(df['title'], rel):
        cand = kobis[kobis['key'] == norm(title)]
        if len(cand):
            gap = (cand['open_dt'] - r).abs().dt.days.fillna(9999)
            best = cand.loc[gap.idxmin()]
            if gap.min() <= 30:
                out.append(best[K_COLS].to_dict() | {'k_gap_days': gap.min()})
                continue
        out.append({c: np.nan for c in K_COLS} | {'k_gap_days': np.nan})
    m = pd.DataFrame(out, index=df.index)
    m.columns = [f'k_{c}' if c in K_COLS else c for c in m.columns]
    return m


K_tr, K_te = match(train), match(test)
for name, df, K in [('train', train, K_tr), ('test', test, K_te)]:
    ok = K['k_max_scrn'].notna()
    print(f'[{name}] 매칭 {ok.sum()}/{len(df)}편 ({ok.mean()*100:.1f}%)')
ok = K_tr['k_max_scrn'].notna()
print(f'train 관객수 100만+ 영화 중 매칭: {(ok & hit).sum()}/{hit.sum()}편')
print(f'train 관객수 1만 미만 영화 중 매칭: {(ok & (y < 10_000)).sum()}/{(y < 10_000).sum()}편')

[train] 매칭 275/600편 (45.8%)
[test] 매칭 111/243편 (45.7%)
train 관객수 100만+ 영화 중 매칭: 109/109편
train 관객수 1만 미만 영화 중 매칭: 5/288편


In [3]:
# 상위 10위에 한 번도 안 든 영화 → 0으로 채우고 플래그
def finalize(K):
    K = K.drop(columns='k_gap_days').copy()
    K['k_in_top10'] = K['k_max_scrn'].notna().astype(int)
    K['k_best_rank'] = K['k_best_rank'].fillna(11)        # 10위 밖
    return K.fillna(0).astype(float)


KF_tr, KF_te = finalize(K_tr).reset_index(drop=True), finalize(K_te).reset_index(drop=True)
display(pd.concat([train[['title', 'release_time', 'box_off_num']], KF_tr], axis=1)
        .sort_values('box_off_num', ascending=False).head(10)
        .style.format({c: '{:,.0f}' for c in ['box_off_num'] + list(KF_tr.columns)}))

,title,release_time,box_off_num,k_open_scrn,k_open_show,k_max_scrn,k_max_show,k_days_top10,k_best_rank,k_in_top10
372,국제시장,2014-12-17,"14,262,766",931,"4,444","1,044","5,015",82,1,1
362,도둑들,2012-07-25,"12,983,841",890,"4,402","1,091","5,475",68,1,1
530,7번방의 선물,2013-01-23,"12,811,435",605,"2,874",866,"4,133",82,1,1
498,암살,2015-07-22,"12,706,663","1,264","6,245","1,519","7,377",66,1,1
460,"광해, 왕이 된 남자",2012-09-13,"12,323,595",688,"3,362","1,001","4,915",98,1,1
122,변호인,2013-12-18,"11,374,879",591,"1,603",925,"4,604",60,1,1
496,설국열차,2013-08-01,"9,350,351",936,"4,765","1,128","5,577",59,1,1
101,관상,2013-09-11,"9,135,806",927,"4,373","1,240","6,124",46,1,1
505,해적: 바다로 간 산적,2014-08-06,"8,666,208",786,"3,970",910,"4,395",67,1,1
476,수상한 그녀,2014-01-22,"8,659,725",577,"2,672","1,027","4,236",71,1,1


## 2. 외부 변수와 관객수의 관계

In [4]:
rel = pd.DataFrame({
    'Spearman (전체)': KF_tr.corrwith(y, method='spearman'),
    'Spearman (상위 10위 진입 영화만)': KF_tr[KF_tr['k_in_top10'] == 1].corrwith(y[KF_tr['k_in_top10'] == 1], method='spearman'),
})
rel.loc['참고: num_staff'] = [train['num_staff'].corr(y, method='spearman'),
                             train.loc[KF_tr['k_in_top10'] == 1, 'num_staff'].corr(y[KF_tr['k_in_top10'] == 1], method='spearman')]
display(rel.style.format('{:.3f}'))
print('→ 큰 영화끼리(상위 10위 진입 영화) 비교할 때 스크린 수가 스태프 수보다 관객수를 훨씬 잘 설명하면, 흥행작 오차를 줄일 수 있음')

,Spearman (전체),Spearman (상위 10위 진입 영화만)
k_open_scrn,0.882,0.837
k_open_show,0.883,0.840
k_max_scrn,0.892,0.893
k_max_show,0.894,0.909
k_days_top10,0.898,0.941
k_best_rank,-0.894,-0.910
k_in_top10,0.841,nan
참고: num_staff,0.695,0.572


→ 큰 영화끼리(상위 10위 진입 영화) 비교할 때 스크린 수가 스태프 수보다 관객수를 훨씬 잘 설명하면, 흥행작 오차를 줄일 수 있음


## 3. 08 방식 + KOBIS 변수
- 부스팅 · 트리 6개: 08 디벨롭 최고(D2) 변수 + KOBIS 변수, log 타깃, 랭크 인코딩은 fold 안에서
- TabPFN v3.5: 06 최종 변수 + KOBIS 변수, sqrt 타깃
- NNLS 가중 평균 (OOF로 가중치 결정)

In [5]:
# ── 08과 같은 원본 1위 코드 변수 ──
def clean_distributor(s):
    s = s.str.replace('(주)', '', regex=False)
    return s.map(lambda x: re.sub(r'[^0-9a-zA-Z가-힣]', '', x))


def get_dis(x):
    for k, v in [('CJ', 'CJ'), ('CGV', 'CJ'), ('쇼박스', '쇼박스'), ('SK', 'SK'), ('리틀빅픽', '리틀빅픽처스'), ('스폰지', '스폰지'),
                 ('싸이더스', '싸이더스'), ('에이원', '에이원'), ('마인스', '마인스'), ('마운틴픽', '마운틴픽처스'), ('디씨드', '디씨드'),
                 ('드림팩트', '드림팩트'), ('메가박스', '메가박스'), ('마운틴', '마운틴')]:
        if k in x:
            return v
    return x


for df in (train, test):
    df['dist_clean'] = clean_distributor(df['distributor']).map(get_dis)


def original_X(tr_df, tr_y, ap_df):
    tmp = tr_df.assign(_y=tr_y.values)
    g_rank = tmp.groupby('genre')['_y'].mean().rank(method='first')
    d_rank = tmp.groupby('dist_clean')['_y'].median().rank(method='first')
    X = pd.DataFrame({'num_rank': ap_df['dist_clean'].map(d_rank).fillna(0).values, 'time': ap_df['time'].values,
                      'num_staff': ap_df['num_staff'].values, 'num_actor': np.log1p(ap_df['num_actor'].values),
                      'genre_rank': ap_df['genre'].map(g_rank).fillna(0).values})
    for r in ['12세 관람가', '15세 관람가', '전체 관람가', '청소년 관람불가']:
        X[f'rat_{r}'] = (ap_df['screening_rat'].values == r).astype(float)
    return X.astype(float)


EXTRA_COLS = ['dir_prev_bfnum', 'is_debut', 'dir_prev_num', '년', '월', 'weekday', 'quarter', 'is_peak',
              'is_15', 'is_all', 'staff_x_rat0', 'staff_x_rat1', 'staff_x_rat2', 'staff_x_rat3', 'dist_count', 'is_big4']
EX_tr, EX_te = make_features(train, test, {'debut': 1, 'debut_mode': 'zero_flag', 'distributor': 1, 'date': 1, 'rating': 1})
EX_tr = pd.concat([EX_tr[EXTRA_COLS].reset_index(drop=True), KF_tr], axis=1)
EX_te = pd.concat([EX_te[EXTRA_COLS].reset_index(drop=True), KF_te], axis=1)

# TabPFN 입력: 06 최종 변수 + KOBIS
PF_tr, PF_te = make_features(train, test, {'debut': 1, 'debut_mode': 'nan', 'distributor': 1, 'date': 1, 'rating': 1})
PF_tr = pd.concat([PF_tr.reset_index(drop=True), KF_tr], axis=1)
PF_te = pd.concat([PF_te.reset_index(drop=True), KF_te], axis=1)

MODELS = {
    'GBM':  lambda: GradientBoostingRegressor(random_state=42),
    'NGB':  lambda: NGBRegressor(random_state=518, verbose=False),
    'LGBM': lambda: LGBMRegressor(random_state=518, verbose=-1),
    'XGB':  lambda: XGBRegressor(random_state=518),
    'CAT':  lambda: CatBoostRegressor(random_state=518, silent=True),
    'RF':   lambda: RandomForestRegressor(random_state=518),
}
print('부스팅 · 트리 변수', original_X(train, y, train).shape[1] + EX_tr.shape[1], '개 | TabPFN 변수', PF_tr.shape[1], '개')

부스팅 · 트리 변수 32 개 | TabPFN 변수 48 개


In [6]:
def run_trees():
    path = os.path.join(CACHE_DIR, 'trees.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    oofs = {m: np.zeros(len(y)) for m in MODELS}; tests = {m: np.zeros(len(test)) for m in MODELS}
    for tr_idx, va_idx in kf5.split(train):
        Xtr = pd.concat([original_X(train.iloc[tr_idx], y.iloc[tr_idx], train.iloc[tr_idx]).reset_index(drop=True),
                         EX_tr.iloc[tr_idx].reset_index(drop=True)], axis=1)
        Xva = pd.concat([original_X(train.iloc[tr_idx], y.iloc[tr_idx], train.iloc[va_idx]).reset_index(drop=True),
                         EX_tr.iloc[va_idx].reset_index(drop=True)], axis=1)
        Xte = pd.concat([original_X(train.iloc[tr_idx], y.iloc[tr_idx], test).reset_index(drop=True), EX_te], axis=1)
        for name, mk in MODELS.items():
            m = mk(); m.fit(Xtr, np.log1p(y.iloc[tr_idx]))
            oofs[name][va_idx] = np.expm1(np.clip(m.predict(Xva), 0, None))
            tests[name] += np.expm1(np.clip(m.predict(Xte), 0, None)) / 5
    res = {'oofs': oofs, 'tests': tests}
    pickle.dump(res, open(path, 'wb'))
    return res


def run_tabpfn():
    path = os.path.join(CACHE_DIR, 'tabpfn.pkl')
    if os.path.exists(path):
        return pickle.load(open(path, 'rb'))
    mk = lambda: TabPFNRegressor.create_default_for_version(ModelVersion.V3_5, device=DEVICE, random_state=42)
    oof = np.zeros(len(y))
    for tr_idx, va_idx in kf5.split(PF_tr):
        m = mk(); m.fit(PF_tr.iloc[tr_idx], np.sqrt(y.iloc[tr_idx]))
        oof[va_idx] = np.square(np.clip(m.predict(PF_tr.iloc[va_idx]), 0, None))
    m = mk(); m.fit(PF_tr, np.sqrt(y))
    res = {'oof': oof, 'test': np.square(np.clip(m.predict(PF_te), 0, None))}
    pickle.dump(res, open(path, 'wb'))
    return res


t0 = time.time()
trees = run_trees()
print(f'부스팅 · 트리 6개 완료 ({time.time()-t0:.0f}초)')
t0 = time.time()
pfn = run_tabpfn()
print(f'TabPFN 완료 ({time.time()-t0:.0f}초)')

부스팅 · 트리 6개 완료 (192초)


TabPFN 완료 (445초)


## 4. 결과 비교 (08 · 06과 같은 5-Fold)

In [7]:
members_oof = {**trees['oofs'], 'TabPFN': pfn['oof']}
members_test = {**trees['tests'], 'TabPFN': pfn['test']}
names = list(members_oof)
O = np.column_stack([members_oof[n] for n in names]); T = np.column_stack([members_test[n] for n in names])
w, _ = nnls(O, y.values.astype(float)); w = w / w.sum()
blend_oof, blend_test = O @ w, T @ w

display(pd.DataFrame({'모델': names, '단독 CV RMSE': [rmse(y, members_oof[n]) for n in names],
                      '흥행작 RMSE': [rmse(y[hit], members_oof[n][hit]) for n in names], 'NNLS 가중치': w})
        .sort_values('단독 CV RMSE').style.format({'단독 CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', 'NNLS 가중치': '{:.3f}'}))

comp = pd.DataFrame([
    {'방법': '06 TabPFN 단독 (KOBIS 없음)', 'CV RMSE': 1_350_295, '흥행작 RMSE': 3_025_307},
    {'방법': '08 최종 가중 평균 (KOBIS 없음) · Public 1,183,942', 'CV RMSE': 1_351_383, '흥행작 RMSE': 3_055_060},
    {'방법': '09 TabPFN 단독 + KOBIS', 'CV RMSE': rmse(y, pfn['oof']), '흥행작 RMSE': rmse(y[hit], pfn['oof'][hit])},
    {'방법': '09 가중 평균 + KOBIS', 'CV RMSE': rmse(y, blend_oof), '흥행작 RMSE': rmse(y[hit], blend_oof[hit])},
])
display(comp.style.format({'CV RMSE': '{:,.0f}', '흥행작 RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE', '흥행작 RMSE'], color='#c6efce'))

,모델,단독 CV RMSE,흥행작 RMSE,NNLS 가중치
6,TabPFN,"363,107","837,828",0.725
3,XGB,"547,103","1,264,402",0.179
5,RF,"549,738","1,276,941",0.096
0,GBM,"615,296","1,427,186",0.000
1,NGB,"622,102","1,445,103",0.000
4,CAT,"645,634","1,496,631",0.000
2,LGBM,"678,894","1,578,394",0.000


,방법,CV RMSE,흥행작 RMSE
0,06 TabPFN 단독 (KOBIS 없음),"1,350,295","3,025,307"
1,"08 최종 가중 평균 (KOBIS 없음) · Public 1,183,942","1,351,383","3,055,060"
2,09 TabPFN 단독 + KOBIS,"363,107","837,828"
3,09 가중 평균 + KOBIS,"357,658","824,752"


In [8]:
top = y.sort_values(ascending=False).index[:12]
old = pickle.load(open('cache_06/M_nan.pkl', 'rb'))['oof'] if os.path.exists('cache_06/M_nan.pkl') else np.full(len(y), np.nan)
cmp = pd.DataFrame({'제목': train.loc[top, 'title'], '실제': y[top], 'TabPFN (KOBIS 없음)': old[top],
                    '09 가중 평균 + KOBIS': blend_oof[top], '개봉일 스크린': KF_tr.loc[top, 'k_open_scrn']})
display(cmp.style.format({c: '{:,.0f}' for c in cmp.columns if c != '제목'}))

,제목,실제,TabPFN (KOBIS 없음),09 가중 평균 + KOBIS,개봉일 스크린
372,국제시장,"14,262,766","3,271,494","11,423,698",931
362,도둑들,"12,983,841","8,438,953","11,059,535",890
530,7번방의 선물,"12,811,435","4,771,265","10,317,307",605
498,암살,"12,706,663","6,364,228","12,562,164","1,264"
460,"광해, 왕이 된 남자","12,323,595","2,016,706","12,293,076",688
122,변호인,"11,374,879","2,311,562","8,460,579",591
496,설국열차,"9,350,351","959,755","10,079,585",936
101,관상,"9,135,806","3,363,345","6,977,635",927
505,해적: 바다로 간 산적,"8,666,208","3,382,569","9,275,225",786
476,수상한 그녀,"8,659,725","4,336,170","11,072,442",577


## 5. 제출 파일

In [9]:
final = np.clip(blend_test, 0, None)
sub = submission.copy(); sub['box_off_num'] = final
sub.to_csv('09_kobis_final.csv', index=False)
print(f'저장: 09_kobis_final.csv (가중 평균 + KOBIS) | 예측 평균 {final.mean():,.0f} · 최대 {final.max():,.0f}')

sub2 = submission.copy(); sub2['box_off_num'] = pfn['test']
sub2.to_csv('09_kobis_tabpfn.csv', index=False)
print('저장: 09_kobis_tabpfn.csv (TabPFN 단독 + KOBIS, 비교용)')
comp.to_csv('09_results.csv', index=False, encoding='utf-8-sig')

저장: 09_kobis_final.csv (가중 평균 + KOBIS) | 예측 평균 646,542 · 최대 12,148,882
저장: 09_kobis_tabpfn.csv (TabPFN 단독 + KOBIS, 비교용)
